### Master of Applied Artificial Intelligence

**Course: TC5035 - Proyecto Integrador**

<img src="https://github.com/Medicenchapin/Proyecto-Integrador/blob/main/assets/logo.png?raw=1" alt="Image Alt Text" width="500"/>


**Other models**

Tutor: Dr. Horario Martinez Alfaro


Team members:
* Ignacio Jose Aguilar Garcia - A00819762
* Alejandro Calderon Aguilar - A01795353
* Ricardo Mar Cupido - A01795394

### 1) ¿El rendimiento del modelo es lo suficientemente bueno para su implementación en producción?

Sí, para un **MVP en producción controlada**. Con base en *Avance5_Equipo25* y *4_other_models*, el clasificador cumple los criterios mínimos y opera **sin riesgo de fuga**. Por gobernanza y madurez interna, la **línea base de despliegue seguirá siendo XGBoost** (estándar histórico de la empresa), mientras la **capa LLM** (modelo final abierto tras pruebas) demostró **mayor claridad descriptiva** y **mejor relación costo/beneficio** para generar guiones anclados en evidencia. Los **drivers TOP-N** (p. ej., *state_name, previous_classification, previous_calls, client_age, network_age_years, banking, arpu_90_days≡ARPU_3M_PROM, minutes_in, validity_average, average_performance, start_using_months, contacts, high_frequency_contacts, plan_postpaid, sn_banking, digital_index_mean, connected_days, charged_days, apps_days, music_gb*; *sale* solo como etiqueta) permiten **trazabilidad y explicabilidad** para los stakeholders. Con esto, el sistema es **implementable** en un **piloto supervisado**.

Quick Wins:
Mantener XGBoost como baseline operativo por su robustez, interpretabilidad (SHAP) y alineación con la gobernanza interna; usar el LLM como capa de generación de guiones y enriquecimiento contextual cuando aporte claridad y evidencia.

Diferentes modelos explorados en `4_other_models` muestran trade-offs: algunos mejoran claridad descriptiva (LLM) y otros rendimiento puro en clasificación (tree-based). El enfoque mixto (XGBoost + LLM) combina trazabilidad y experiencia conversacional.

### 2) ¿Existe margen para mejorar aún más el rendimiento?

Sí, en dos frentes. **Modelo**: *tuning* fino de XGBoost (regularización, *learning rate*, *early stopping*), **calibración de probabilidades**, revisión de desbalance, y *feature engineering* guiado por los drivers globales. **LLM**: fortalecer *prompting* (plantillas por segmento), validar *JSON espejo* contra *schema*, probar *few-shot* y, si aplica, **RAG** con catálogos/ofertas vigentes; además, optimizar **latencia y costo** con *max_tokens* y *temperature* controlados. El **impacto en negocio** se materializa al reducir **minutos de preparación del guion por cliente**: si (N_{\text{clientes/día}}=\frac{\text{minutos totales}}{\text{min/cliente}}), entonces una reducción (\Delta t) eleva (N); el ingreso incremental se aproxima por (\Delta \text{Ventas} \approx \Delta N \times \text{tasa de contacto} \times \text{tasa de conversión} \times \text{ticket medio}).


### 3) ¿Cuáles serían las recomendaciones clave para poder implementar la solución?

(1) **API única** (FastAPI) que exponga *scoring* de XGBoost y *prompting* del LLM; (2) **contenedores** (Docker) y *autoscaling* por campaña; (3) **versionado** en *model registry/feature store* (artefactos, semillas, *hash* de *playbooks*); (4) **observabilidad**: latencia, errores, *retries/backoff*, % de respuestas LLM válidas (JSON), y **cost caps** por 1K tokens; (5) **guardrails**: validación de esquema, anonimización, listas blancas de campos, *circuit breakers* y *fallbacks* (guion mínimo estándar) ante fallas del LLM; (6) **experimentación**: *shadow* + **A/B** por campaña con KPIs de negocio (uplift de conversión y **reducción de min/cliente**); (7) **despliegue gradual** empezando por **Prepago→Pospago**, ampliando por oleadas; (8) **seguridad y cumplimiento** (PII, retención) y *runbooks* de **rollback**.

 Observabilidad y despliegue (mínimos a implementar)
 
- Monitoreo: latencia, error rate, % JSON válido, token usage y costo, fallback rate, drift de features (mean, std), y alertas automáticas cuando se superen umbrales.


### 4) ¿Qué tareas / procedimientos son accionables para las partes interesadas (stakeholders)?

**Comercial/Telemarketing:** definir ofertas y cohortes de prueba; fijar métrica primaria y umbral operativo; lanzar **piloto** con muestra **control** vs **tratamiento**; medir discurso, **min/cliente** y conversión. **Operaciones:** capacitar agentes en lectura de *bullets* por driver; instrumentar captura de tiempos y feedback; asegurar adherencia de guion. **Datos/TI:** desplegar API, *batch jobs* y *scheduler*; contenedores por campaña; monitoreo (latencia, *timeouts*, % JSON válido, costos); **modelo XGBoost** y *playbooks* versionados. **Compliance/Legal:** revisar variables sensibles y textos; políticas de anonimización/retención. **Finanzas:** seguimiento de **ROI** (costo LLM vs margen incremental), umbrales de rentabilidad por campaña. **Dirección/PMO:** gobernanza de cambios, cadencia de *reviews* (semanales) y *go/no-go* por oleada. Con este *plan operativo* el MVP es **viable**: se preserva XGBoost por alineación organizacional y se capitaliza el **LLM** para **acelerar el *time-to-pitch*** y aumentar el **throughput** comercial, siempre con trazabilidad basada en **drivers SHAP**.


### Próximos pasos sugeridos (rápidos, en 1-2 sprints)
1. Implementar pipeline de tuning de XGBoost reproducible y almacenar artifacts con metadatos.
2. Implementar un endpoint FastAPI mínimo o un Dockerfile; correr smoke tests de latencia y schema.
3. Preparar A/B o shadow run en una campaña pequeña: comparar baseline XGBoost vs XGBoost+LLM en métricas de negocio (conversión, min/cliente).

### 5) ¿Cuál es la plataforma más adecuada para implementar la solución considerando los modelos probados?

**Infraestructura propia de la empresa completamente local**, aprovechando que la organización cuenta con recursos internos suficientes. Considerando que probamos **Mistral** (`koesn/mistral-7b-instruct`) y **Hermes** (`cas/nous-hermes-2-mistral-7b-dpo`) localmente con **Ollama**, estos modelos ya están validados para ejecutarse en servidores internos. La estrategia será: **servidores locales** con GPUs para ejecutar modelos LLM, helpers SHAP y lógica de prompting; **almacenamiento local** para campañas y artifacts; **APIs internas** para comunicación entre componentes. Esto elimina completamente dependencias externas y costos recurrentes de APIs cloud. Para el **MVP controlado**, usar infraestructura 100% local maximiza control sobre datos y modelos.

### 6) ¿Qué factores técnicos y de costo justifican usar infraestructura completamente local?

**Factores técnicos avanzados**: 
(1) **Control total de datos** - información sensible de clientes permanece 100% dentro del perímetro empresarial, cumpliendo estrictas políticas de privacidad; (2) **Modelos validados localmente** - Mistral (`koesn/mistral-7b-instruct`) y Hermes (`cas/nous-hermes-2-mistral-7b-dpo`) ya probados exitosamente con Ollama en notebooks, eliminando riesgo de migración; (3) **Latencia ultra-optimizada** - tiempo de respuesta <200ms vs >500ms de APIs externas, sin dependencias de ancho de banda o conectividad; (4) **Integración nativa profunda** - conexión directa con CRM, bases de datos y sistemas telefónicos sin middleware adicional; (5) **Personalización ilimitada** - capacidad completa de fine-tuning, ajuste de parámetros y optimización específica para casos de telemarketing.


### 7) ¿Cuáles son las ventajas operativas y consideraciones técnicas de esta estrategia completamente local?

**Ventajas operativas detalladas:**
- **Seguridad empresarial**: Datos críticos nunca abandonan instalaciones, cumpliendo políticas corporativas más estrictas y eliminando riesgos de data breach externos
- **Independencia operativa total**: Cero dependencias de conectividad internet, estabilidad de APIs externas o cambios en términos de servicio de terceros
- **Control de calidad**: Capacidad de monitorear, ajustar y optimizar modelos en tiempo real según métricas específicas de conversión y satisfacción de agentes
- **Personalización profunda**: Fine-tuning de modelos con datos históricos propios, ajuste de prompts por segmento de cliente, y optimización continua basada en feedback del call center
- **Compliance simplificado**: Menor complejidad regulatoria al evitar transferencias de datos, auditorías de terceros y certificaciones de proveedores externos

**Consideraciones técnicas específicas:**
- **Especificaciones de hardware**: Servidores con GPU NVIDIA RTX 4090/A100 (mínimo 24GB VRAM) para ejecutar Mistral-7B eficientemente, almacenamiento SSD de alta velocidad (1TB+) para modelos y datos de campaña
- **Arquitectura de red interna**: Conectividad Gigabit entre servidores GPU y sistemas CRM, balanceadores de carga para alta disponibilidad, y segmentación de red para aislar componentes críticos
- **Monitoreo y observabilidad**: Implementación de métricas propias (latencia por request, throughput de tokens, uso de GPU, tasa de error), dashboards en tiempo real, y alertas automáticas para degradación de performance
- **Estrategia de backup**: Replicación automática de modelos y configuraciones, snapshots regulares de datos de campaña, y procedimientos de disaster recovery sin dependencias cloud
- **Gestión de versiones**: Control de versiones de modelos LLM, rollback automático ante degradación de performance, y testing A/B interno entre versiones de modelos
